# Absolute-Residual Reviewer Experiments

## Setup

The plotting and table helpers below come from `utility.graphing_tools`. They are designed to work with the newer experiment runner outputs, including coordinate-level summaries.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from utility.exps import (
    run_abs_res_synthetic_experiment,
    run_abs_res_dependent_gaussian_experiment,
)
from utility.graphing_tools import (
    load_experiment_dataframes,
    save_experiment_dataframes,
    reviewer_summary_table,
    coordinate_length_table,
    summarize_infinite_volume_rate,
    plot_reviewer_metric_grid,
    plot_coordinate_length_profile,
    plot_metric_sweep,
)

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

DATA_DIR = Path('data/reviewer_exps') / 'absolute_residual'
DATA_DIR.mkdir(parents=True, exist_ok=True)

METHODS = ['TSCP_R', 'Point_CHR', 'Unscaled', 'Empirical_copula']
DISPLAY_METHODS = ['TSCP (Ours)', 'Point CHR', 'Unscaled Max', 'Emp. Copula']

ALPHA = 0.1
TARGET_COVERAGE = 1 - ALPHA
DIM = 10
N_FEATURES = 10
N_INFORMATIVE = 10

print(f'Data folder: {DATA_DIR.resolve()}')

## Run Controls

Use smoke mode first to verify the workflow quickly. For advisor-facing or rebuttal-quality runs, set `RUN_FULL_EXPERIMENTS = True` and `FORCE_RERUN = True`.

In [ ]:
RUN_FULL_EXPERIMENTS = True
FORCE_RERUN = True

TRIALS = 200 if RUN_FULL_EXPERIMENTS else 3
N_SYNTHETIC_OBSERVATIONS = 8000 if RUN_FULL_EXPERIMENTS else 500
SAMPLE_LIST = [30, 50, 100, 300, 500] if RUN_FULL_EXPERIMENTS else [20, 40]
SMALL_SAMPLE_LIST = [10, 20, 30, 50, 100] if RUN_FULL_EXPERIMENTS else [10, 20]
CORRELATIONS = [0.0, 0.3, 0.6, 0.9] if RUN_FULL_EXPERIMENTS else [0.0, 0.6]
NOISE_RATIOS = [1, 2, 5, 10, 20] if RUN_FULL_EXPERIMENTS else [1, 10]
ALPHA_LIST = [0.05, 0.1, 0.2] if RUN_FULL_EXPERIMENTS else [0.1, 0.2]

print({
    'trials': TRIALS,
    'n_synthetic_observations': N_SYNTHETIC_OBSERVATIONS,
    'sample_list': SAMPLE_LIST,
    'correlations': CORRELATIONS,
    'noise_ratios': NOISE_RATIOS,
    'alpha_list': ALPHA_LIST,
})

In [ ]:
def result_to_tables(result):
    return {
        'trial': result.trial_results,
        'summary': result.summary_results,
        'coordinate_trial': result.coordinate_trial_results,
        'coordinate_summary': result.coordinate_summary_results,
    }


def combine_result_tables(results):
    names = ['trial', 'summary', 'coordinate_trial', 'coordinate_summary']
    return {
        name: pd.concat([result_to_tables(result)[name] for result in results], ignore_index=True)
        for name in names
    }


def load_or_run(prefix, run_fn):
    required = ['trial', 'summary', 'coordinate_trial', 'coordinate_summary']
    if not FORCE_RERUN:
        loaded = load_experiment_dataframes(DATA_DIR, prefix, names=required)
        if all(name in loaded for name in required):
            print(f'Loaded saved tables for {prefix}')
            return loaded

    print(f'Running {prefix}...')
    tables = run_fn()
    paths = save_experiment_dataframes(tables, DATA_DIR, prefix)
    print('Saved:')
    for name, path in paths.items():
        print(f'  {name}: {path}')
    return tables


def geometric_noise_levels(ratio, dim=DIM):
    return np.geomspace(float(ratio), 1.0, dim)

## Experiment 1: Dependent Gaussian Noise

**Purpose.** The current paper uses independent noise across outcome coordinates. This experiment directly addresses the reviewer request by adding dependence while preserving heterogeneous marginal noise scales.

**Design.** We use Gaussian target noise with marginal scales `[10, 9, ..., 1]` and equicorrelation `rho`. The main outputs are coverage, residual-space volume, runtime, and saved coordinate-wise summaries.

**Interpretation target.** If the story is stable, `TSCP_R` should keep near-nominal coverage and remain efficient even when dependence increases. `Empirical_copula` may become small but can under-cover because it lacks the same finite-sample guarantee.

In [ ]:
def run_dependent_gaussian_experiment():
    results = []
    for rho in CORRELATIONS:
        result = run_abs_res_dependent_gaussian_experiment(
            dim_list=[DIM],
            sample_list=SAMPLE_LIST,
            alpha_list=[ALPHA],
            trials=TRIALS,
            methods=METHODS,
            n_train=int(0.8 * (N_SYNTHETIC_OBSERVATIONS)), n_test=(N_SYNTHETIC_OBSERVATIONS) - int(0.8 * (N_SYNTHETIC_OBSERVATIONS)),
            n_features=N_FEATURES,
            n_informative=N_INFORMATIVE,
            correlation=rho,
            correlation_structure='equicorrelated',
            output_dir=None,
            experiment_name=f'dependent_gaussian_rho_{rho}',
        )
        results.append(result)
    return combine_result_tables(results)


dependent_tables = load_or_run('dependent_gaussian', run_dependent_gaussian_experiment)
dependent_summary = dependent_tables['summary']
dependent_coordinate_summary = dependent_tables['coordinate_summary']

display(reviewer_summary_table(
    dependent_summary,
    group_cols=['correlation', 'n_cal'],
    methods=DISPLAY_METHODS,
    target_coverage=TARGET_COVERAGE,
).head(24))

In [ ]:
rho_to_show = 0.6 if 0.6 in set(dependent_summary['correlation']) else dependent_summary['correlation'].max()
plot_df = dependent_summary[dependent_summary['correlation'].eq(rho_to_show)]

fig, axes = plot_reviewer_metric_grid(
    plot_df,
    x='n_cal',
    methods=DISPLAY_METHODS,
    target_coverage=TARGET_COVERAGE,
    log_x=True,
    title=f'Dependent Gaussian Noise, rho={rho_to_show}',
)
plt.show()

n_cal_to_show = 100 if 100 in set(dependent_summary['n_cal']) else dependent_summary['n_cal'].max()
sweep_df = dependent_summary[dependent_summary['n_cal'].eq(n_cal_to_show)]

fig, ax = plot_metric_sweep(
    sweep_df,
    x='correlation',
    metric='test_coverage_avg',
    methods=DISPLAY_METHODS,
    target_coverage=TARGET_COVERAGE,
    log_y=False,
    plot_kind='bar',
    title=f'Coverage by Dependence Strength, n_cal={n_cal_to_show}',
)
plt.show()

fig, ax = plot_metric_sweep(
    sweep_df,
    x='correlation',
    metric='coverage_vol_avg',
    methods=DISPLAY_METHODS,
    plot_kind='bar',
    title=f'Volume by Dependence Strength, n_cal={n_cal_to_show}',
)
plt.show()

## Experiment 2: Coordinate-Wise Length Profiles

**Purpose.** The paper currently reports aggregate volume, but one selling point of TSCP is coordinate-wise interpretability. This experiment reports the average interval length coordinate by coordinate.

**Design.** We reuse Experiment 1. For absolute residual scores, the stored coordinate threshold is a residual-space half-width, so the displayed outcome-space interval length is `2 * threshold`.

**Interpretation target.** `TSCP_R` should adapt to heterogeneous coordinate scales. `Unscaled` should look nearly flat across coordinates because it uses a single max residual threshold.

In [ ]:
coord_plot_df = dependent_coordinate_summary[
    dependent_coordinate_summary['correlation'].eq(rho_to_show)
    & dependent_coordinate_summary['n_cal'].eq(n_cal_to_show)
].copy()

fig, ax = plot_coordinate_length_profile(
    coord_plot_df,
    methods=DISPLAY_METHODS,
    noise_levels=np.arange(DIM, 0, -1),
    coordinate_scale=2.0,
    title=f'Coordinate-Wise Outcome Lengths, rho={rho_to_show}, n_cal={n_cal_to_show}',
)
plt.show()

coord_table = coordinate_length_table(
    coord_plot_df,
    methods=DISPLAY_METHODS,
    noise_levels=np.arange(DIM, 0, -1),
    coordinate_scale=2.0,
    value_format=lambda value: f'{value:.2f}',
)
display(coord_table)

save_experiment_dataframes(
    {'coordinate_length_profile_table': coord_table},
    DATA_DIR,
    'dependent_gaussian',
)

## Experiment 3: Heterogeneity-Severity Sweep

**Purpose.** The paper contrasts homogeneous Gaussian noise with one heterogeneous Gaussian setting. This sweep fills the gap between those two endpoints.

**Design.** We vary the marginal noise ratio `max_noise / min_noise` while keeping `d=10`. Ratio 1 corresponds to homogeneous noise; larger ratios increasingly favor coordinate-wise standardization.

**Interpretation target.** `Unscaled` should be strongest near ratio 1, while `TSCP_R` should improve relative to `Unscaled` as heterogeneity increases.

In [ ]:
def run_heterogeneity_sweep_experiment():
    results = []
    for ratio in NOISE_RATIOS:
        result = run_abs_res_synthetic_experiment(
            dim_list=[DIM],
            sample_list=[n_cal_to_show],
            alpha_list=[ALPHA],
            noise_type='Gaussian',
            noises_list=[geometric_noise_levels(ratio)],
            trials=TRIALS,
            methods=METHODS,
            n_train=int(0.8 * (N_SYNTHETIC_OBSERVATIONS)), n_test=(N_SYNTHETIC_OBSERVATIONS) - int(0.8 * (N_SYNTHETIC_OBSERVATIONS)),
            n_features=N_FEATURES,
            n_informative=N_INFORMATIVE,
            output_dir=None,
            experiment_name=f'heterogeneity_ratio_{ratio}',
        )
        tables = result_to_tables(result)
        for df in tables.values():
            df['noise_ratio'] = ratio
        results.append(tables)
    names = ['trial', 'summary', 'coordinate_trial', 'coordinate_summary']
    return {name: pd.concat([tables[name] for tables in results], ignore_index=True) for name in names}


heterogeneity_tables = load_or_run('heterogeneity_sweep', run_heterogeneity_sweep_experiment)
heterogeneity_summary = heterogeneity_tables['summary']

display(reviewer_summary_table(
    heterogeneity_summary,
    group_cols=['noise_ratio', 'n_cal'],
    methods=DISPLAY_METHODS,
    target_coverage=TARGET_COVERAGE,
))

In [ ]:
fig, ax = plot_metric_sweep(
    heterogeneity_summary,
    x='noise_ratio',
    metric='test_coverage_avg',
    methods=DISPLAY_METHODS,
    target_coverage=TARGET_COVERAGE,
    log_y=False,
    plot_kind='bar',
    title=f'Coverage by Noise Heterogeneity, n_cal={n_cal_to_show}',
)
plt.show()

fig, ax = plot_metric_sweep(
    heterogeneity_summary,
    x='noise_ratio',
    metric='coverage_vol_avg',
    methods=DISPLAY_METHODS,
    plot_kind='bar',
    title=f'Volume by Noise Heterogeneity, n_cal={n_cal_to_show}',
)
plt.show()

## Experiment 4: Alpha Sensitivity

**Purpose.** The current paper emphasizes `alpha=0.1`. This small robustness check verifies that the same qualitative behavior holds across common coverage targets.

**Design.** We use dependent heterogeneous Gaussian noise with `rho=0.6`, fixed calibration size, and `alpha in {0.05, 0.1, 0.2}` in full mode.

**Interpretation target.** Methods with finite-sample validity should track the target coverage across alpha values, while efficiency differences remain visible in volume.

In [ ]:
def run_alpha_sensitivity_experiment():
    result = run_abs_res_dependent_gaussian_experiment(
        dim_list=[DIM],
        sample_list=[n_cal_to_show],
        alpha_list=ALPHA_LIST,
        trials=TRIALS,
        methods=METHODS,
        n_train=int(0.8 * (N_SYNTHETIC_OBSERVATIONS)), n_test=(N_SYNTHETIC_OBSERVATIONS) - int(0.8 * (N_SYNTHETIC_OBSERVATIONS)),
        n_features=N_FEATURES,
        n_informative=N_INFORMATIVE,
        correlation=0.6,
        correlation_structure='equicorrelated',
        output_dir=None,
        experiment_name='alpha_sensitivity',
    )
    return result_to_tables(result)


alpha_tables = load_or_run('alpha_sensitivity', run_alpha_sensitivity_experiment)
alpha_summary = alpha_tables['summary'].copy()
alpha_summary['target_coverage'] = 1 - alpha_summary['alpha']

display(reviewer_summary_table(
    alpha_summary,
    group_cols=['alpha', 'n_cal'],
    methods=DISPLAY_METHODS,
    target_coverage=None,
))

In [ ]:
fig, ax = plot_metric_sweep(
    alpha_summary,
    x='alpha',
    metric='test_coverage_avg',
    methods=DISPLAY_METHODS,
    target_coverage=None,
    log_y=False,
    plot_kind='bar',
    title=f'Coverage by Alpha, n_cal={n_cal_to_show}',
)

for alpha in sorted(alpha_summary['alpha'].unique()):
    ax.axhline(1 - alpha, color='gray', linestyle=':', linewidth=1)
plt.show()

fig, ax = plot_metric_sweep(
    alpha_summary,
    x='alpha',
    metric='coverage_vol_avg',
    methods=DISPLAY_METHODS,
    target_coverage=None,
    plot_kind='bar',
    title=f'Volume by Alpha, n_cal={n_cal_to_show}',
)
plt.show()

## Export Index

In [ ]:
export_index = pd.DataFrame({
    'file': [path.name for path in sorted(DATA_DIR.glob('*.csv'))],
    'path': [str(path.resolve()) for path in sorted(DATA_DIR.glob('*.csv'))],
})
display(export_index)